# Kaggle 2/2: train and evaluate (GPU, about 1 hour)

Runs **steps 2 to 7**: detector training, detector-only evaluation, hard negative mining,
classifier training, two-stage evaluation and predictions on the test images.

**Settings:** **Add Input** -> Your Work -> the *Kaggle 1/2* notebook. Accelerator **GPU P100** (or T4),
Internet **On**. Then **Save Version, Save & Run All (Commit)**.

In [ ]:
import glob, zipfile
!git clone -q --depth 1 https://github.com/Ak2005github/Mitotic-Figure-Detection-in-Histopathology-Images.git /kaggle/working/repo
!pip install -q ultralytics tifffile imagecodecs
zip_path = glob.glob('/kaggle/input/**/midog_yolo.zip', recursive=True)[0]
zipfile.ZipFile(zip_path).extractall('/tmp')
# point data.yaml at the unpacked folder
p = '/tmp/midog_yolo/data.yaml'
lines = open(p).read().splitlines()
open(p, 'w').write('\n'.join('path: /tmp/midog_yolo' if l.startswith('path:') else l for l in lines) + '\n')
%cd /kaggle/working/repo

In [ ]:
DATA = '/tmp/midog_yolo'
DET = 'runs/detector/yolov8s/weights/best.pt'
CLS = 'models/classifier_effnet_b0.pt'
!python scripts/02_train_detector.py --data-dir {DATA} 

In [ ]:
!python scripts/03_evaluate_detector.py --weights {DET} --data-dir {DATA} --out results/detector
!python scripts/04_mine_hard_negatives.py --weights {DET} --data-dir {DATA}
!python scripts/05_train_classifier.py --data-dir {DATA} --candidates results/detector/candidates.pkl --out {CLS}
!python scripts/06_evaluate_two_stage.py --data-dir {DATA} --candidates results/detector/candidates.pkl --classifier {CLS} --out results/two_stage
!python scripts/07_predict.py --detector {DET} --classifier {CLS} --from-metrics results/two_stage/metrics.json --source {DATA}/full/test --out results/predictions

In [ ]:
# keep weights + results in the notebook output
import shutil, os
for d in ('runs', 'models', 'results'):
    shutil.copytree(d, f'/kaggle/working/{d}', dirs_exist_ok=True)
if os.path.exists('/kaggle/working/results/detector/candidates.pkl'):
    os.remove('/kaggle/working/results/detector/candidates.pkl')  # large cache, not needed
shutil.rmtree('/kaggle/working/repo')
!du -sh /kaggle/working/*